## Tahap 2 (Babak 2) — Peta Kesetiaan Representasi: layer x komponen x prompt, sekali jalan

Konteks: `notes/research_question/04_pivot3_representation_fidelity.md` (S13.4 tiga babak),
temuan sejauh ini: `notes/findings/` 01-04.

**Pertanyaan notebook ini:** temuan 01-04 (sinyal lemah/berisik/timpang) itu diukur dengan
SATU cara baca: token terakhir, residual stream, satu template prompt. Apakah lemahnya itu
sifat modelnya, atau sifat titik tempel "elektroda" kita? Di sini kita pasang elektroda di
SEMUA tempat dan bikin peta: di lokasi mana kesetiaan (rho lawan jarak opini survei asli)
paling tinggi?

Tiga sumbu yang divariasikan dalam SATU pass GPU:

1. **Layer** — 33 titik residual stream (kayak notebook 07, tapi sekarang lengkap per tipe).
2. **Komponen** — residual vs **attention head** (32 layer x 32 head = 1024 titik baca,
   diambil dari input `o_proj` — titik baca yang SAMA dengan llm-opinions, cuma pakai
   forward hook PyTorch biasa, bukan PyVene) vs **output FFN/MLP** (32 titik).
3. **Prompt/posisi** — 4 varian template per kelompok (kritik construct-validity
   arXiv:2601.18486: satu template = satu operasionalisasi doang). Varian T3 diakhiri
   `Answer:` sehingga posisi baca last-token = "setelah Answer:" ala llm-opinions.

Penggaris SAMA untuk semua titik: Spearman rho antara jarak-embedding antar kelompok vs
`group_real_dist` (jarak Wasserstein distribusi jawaban survei asli), dihitung PER TIPE
(pelajaran finding 02: pooled nutupin ketimpangan antar tipe).

**Patching (sumbu kausal) sengaja TIDAK di sini** — desainnya bergantung hasil peta ini.


## Sebelum jalan: setting Kaggle

1. **Accelerator**: GPU T4 x2. **Internet: On**.
2. **Attach dataset** `opinionqa_intersectional.csv` (sama dengan notebook 07/08).
3. Kalau sesi ini bekas crash/OOM: **RESTART SESSION** dulu (jangan cuma re-run cell).
4. **SETELAH SELESAI (PENTING — kali ini jangan lupa):** download dari
   `/kaggle/working/tahap2_peta_kesetiaan/`:
   - `peta_kesetiaan_full.csv` (WAJIB — seluruh peta, kecil)
   - `peta_kesetiaan_top.csv` (WAJIB — ringkasan lokasi terbaik)
   - `emb_resid.npz`, `emb_heads.npz`, `emb_mlp.npz` (masing-masing ~350MB — download
     minimal `emb_heads.npz` biar analisis lanjutan head-level bebas GPU)
   - semua `.png`

Estimasi waktu: load model ~5-10 menit, ekstraksi 676 prompt ~30-45 menit, analisis CPU
~5-10 menit. Total sesi ~1 jam.


In [ ]:
!pip install -q -U "transformers>=4.44" accelerate scipy scikit-learn tqdm

In [ ]:
import os, sys, gc, glob, ast
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import wasserstein_distance, spearmanr
from sklearn.metrics import pairwise_distances
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# Pelajaran OOM #2 (notebook 08): traceback crash lama bisa nahan tensor GPU di kernel.
sys.last_traceback = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    for d in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(d)
        print(f"GPU {d}: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
        if free / total < 0.9:
            print(f"  PERINGATAN: GPU {d} tidak kosong -> RESTART SESSION dulu sebelum load model!")


In [ ]:
MODEL_PATH = "mistralai/Mistral-7B-v0.1"

_candidates = glob.glob("/kaggle/input/**/opinionqa_intersectional.csv", recursive=True)
if _candidates:
    DATA_PATH = _candidates[0]
elif os.path.exists("opinionqa_intersectional.csv"):
    DATA_PATH = "opinionqa_intersectional.csv"
else:
    raise FileNotFoundError(
        "opinionqa_intersectional.csv tidak ketemu. Attach dataset Kaggle-nya dulu."
    )
print("Pakai data dari:", DATA_PATH)

N_QKEYS_FOR_WD = 300   # sampel pertanyaan bersama per pasangan sel (sama dgn notebook 07)
RANDOM_SEED = 42

OUT_DIR = "/kaggle/working/tahap2_peta_kesetiaan"
os.makedirs(OUT_DIR, exist_ok=True)
print("OUT_DIR:", OUT_DIR)


## 1. Data & metadata sel irisan (sama persis dengan notebook 07)

In [ ]:
df = pd.read_csv(DATA_PATH)

def _parse_list(x):
    return ast.literal_eval(x) if isinstance(x, str) else x

df["responses"] = df["responses"].apply(_parse_list)
df["ordinal"] = df["ordinal"].apply(_parse_list)
df["options"] = df["options"].apply(_parse_list)
df["group_key"] = df["attribute"] + " :: " + df["group"]

GROUP_KEYS = sorted(df["group_key"].unique().tolist())
n_g = len(GROUP_KEYS)

group_meta = {}
for gk in GROUP_KEYS:
    attr_type, group_str = gk.split(" :: ", 1)
    v1, v2 = group_str.split(" | ", 1)
    group_meta[gk] = {"attr_type": attr_type, "v1": v1, "v2": v2}

attr_types = np.array([group_meta[gk]["attr_type"] for gk in GROUP_KEYS])
TYPES = sorted(set(attr_types.tolist()))
type_idx = {t: np.where(attr_types == t)[0] for t in TYPES}

print(f"{n_g} sel irisan, {len(TYPES)} tipe kombinasi:")
for t in TYPES:
    print(f"  {t}: {len(type_idx[t])} sel")


## 2. Jarak-asli antar sel (`group_real_dist`) — penggaris untuk semua titik baca

In [ ]:
resp_lookup = {
    (gk, qk): (resp, ordv)
    for gk, qk, resp, ordv in zip(df["group_key"], df["qkey"], df["responses"], df["ordinal"])
}
qkeys_by_group = df.groupby("group_key")["qkey"].apply(set).to_dict()

rng = np.random.default_rng(RANDOM_SEED)
group_real_dist = np.full((n_g, n_g), np.nan)

for i in tqdm(range(n_g), desc="Jarak-asli antar sel"):
    gi = GROUP_KEYS[i]
    for j in range(i, n_g):
        if i == j:
            group_real_dist[i, j] = 0.0
            continue
        gj = GROUP_KEYS[j]
        shared = qkeys_by_group.get(gi, set()) & qkeys_by_group.get(gj, set())
        if not shared:
            continue
        shared = list(shared)
        if len(shared) > N_QKEYS_FOR_WD:
            idx = rng.choice(len(shared), size=N_QKEYS_FOR_WD, replace=False)
            shared = [shared[k] for k in idx]
        wds = []
        for qk in shared:
            respA, ordA = resp_lookup[(gi, qk)]
            respB, ordB = resp_lookup[(gj, qk)]
            if len(ordA) != len(ordB):
                continue
            wds.append(wasserstein_distance(ordA, ordB, u_weights=respA, v_weights=respB))
        if wds:
            group_real_dist[i, j] = group_real_dist[j, i] = float(np.mean(wds))

iu_all = np.triu_indices(n_g, k=1)
print("NaN pairs:", int(np.isnan(group_real_dist[iu_all]).sum()), "dari", len(iu_all[0]))


## 3. Multi-cue: 4 varian prompt per sel

Kenapa 4: kritik construct-validity (arXiv:2601.18486) — satu template itu cuma satu cara
mengoperasionalkan "identitas kelompok". Kalau peta kesetiaannya berubah drastis antar
template, sinyalnya rapuh (artefak template), bukan properti model.

- **T0** orang-ketiga deklaratif (identik dgn notebook 07 — jadi jembatan ke finding 01-04)
- **T1** orang-pertama ("I am ...")
- **T2** profil terstruktur (bullet)
- **T3** format QA yang diakhiri jawaban demografis — last token = **"setelah Answer:"**,
  posisi baca yang divalidasi llm-opinions untuk readout orde-pertama mereka.


In [ ]:
ATTR_LABELS = {
    "RACExRELIG":       ("race", "religion"),
    "RACExPOLPARTY":    ("race", "political party affiliation"),
    "RACExPOLIDEOLOGY": ("race", "political ideology"),
    "RELIGxPOLPARTY":   ("religion", "political party affiliation"),
    "EDUCATIONxINCOME": ("highest level of education", "household income"),
    "AGExPOLPARTY":     ("age group", "political party affiliation"),
}

def make_templates(attr_type, v1, v2):
    l1, l2 = ATTR_LABELS[attr_type]
    return [
        f"This survey respondent's {l1} is {v1} and their {l2} is {v2}.",
        f"I am answering this survey. My {l1} is {v1} and my {l2} is {v2}.",
        f"Survey respondent profile:\n- {l1}: {v1}\n- {l2}: {v2}\nThis respondent is about to answer opinion questions.",
        f"Question: What is this survey respondent's {l1} and {l2}?\nAnswer: Their {l1} is {v1} and their {l2} is {v2}.",
    ]

N_TEMPLATES = 4
prompts_by_template = {t: {} for t in range(N_TEMPLATES)}
for gk in GROUP_KEYS:
    m = group_meta[gk]
    for t, p in enumerate(make_templates(m["attr_type"], m["v1"], m["v2"])):
        prompts_by_template[t][gk] = p

print(f"Total prompt: {N_TEMPLATES} x {n_g} = {N_TEMPLATES * n_g}\n")
gk0 = GROUP_KEYS[0]
for t in range(N_TEMPLATES):
    print(f"--- T{t} ---")
    print(prompts_by_template[t][gk0])
    print()


## 4. Load model & ekstraksi 3 komponen sekaligus

Satu forward pass memotret, di **token terakhir**:

- **residual**: `hidden_states` (33 x 4096) — 1 snapshot awal + 32 pasca-block
- **head**: input `o_proj` per layer (concat 32 head x 128 dim, SEBELUM dicampur o_proj) —
  titik baca yang sama dgn llm-opinions (`self_attn.o_proj.input`), di-reshape [head, 128]
- **mlp**: output blok FFN per layer (32 x 4096)


In [ ]:
print(f"Loading tokenizer & model: {MODEL_PATH}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, device_map="balanced", low_cpu_mem_usage=True
)
model.eval()

NUM_LAYERS = model.config.num_hidden_layers
NUM_HEADS = model.config.num_attention_heads
HIDDEN = model.config.hidden_size
HEAD_DIM = HIDDEN // NUM_HEADS
print(f"{NUM_LAYERS} layer, {NUM_HEADS} head x {HEAD_DIM} dim, hidden {HIDDEN}")
if torch.cuda.is_available():
    for d in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(d)
        print(f"GPU {d} setelah load: {free/1e9:.1f} GB free")


In [ ]:
_capture = {}

def _oproj_prehook(layer_idx):
    def fn(module, args):
        _capture[("head", layer_idx)] = args[0][0, -1, :].detach().float().cpu()
    return fn

def _mlp_hook(layer_idx):
    def fn(module, args, output):
        _capture[("mlp", layer_idx)] = output[0, -1, :].detach().float().cpu()
    return fn

handles = []
for li, layer in enumerate(model.model.layers):
    handles.append(layer.self_attn.o_proj.register_forward_pre_hook(_oproj_prehook(li)))
    handles.append(layer.mlp.register_forward_hook(_mlp_hook(li)))
print(f"{len(handles)} hook terpasang.")

@torch.no_grad()
def extract_all(prompt):
    _capture.clear()
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model(**inputs, output_hidden_states=True)
    resid = torch.stack(out.hidden_states, dim=0)[:, 0, -1, :].float().cpu().numpy()
    heads = np.stack([_capture[("head", li)].numpy().reshape(NUM_HEADS, HEAD_DIM)
                      for li in range(NUM_LAYERS)])
    mlp = np.stack([_capture[("mlp", li)].numpy() for li in range(NUM_LAYERS)])
    return resid, heads, mlp

resid_all = np.zeros((N_TEMPLATES, n_g, NUM_LAYERS + 1, HIDDEN), dtype=np.float16)
heads_all = np.zeros((N_TEMPLATES, n_g, NUM_LAYERS, NUM_HEADS, HEAD_DIM), dtype=np.float16)
mlp_all   = np.zeros((N_TEMPLATES, n_g, NUM_LAYERS, HIDDEN), dtype=np.float16)

for t in range(N_TEMPLATES):
    for i, gk in enumerate(tqdm(GROUP_KEYS, desc=f"Ekstraksi T{t}")):
        r, h, m = extract_all(prompts_by_template[t][gk])
        resid_all[t, i], heads_all[t, i], mlp_all[t, i] = r, h, m
    torch.cuda.empty_cache()

for h_ in handles:
    h_.remove()
print("Ekstraksi selesai, hook dilepas.")

gkeys_arr = np.array(GROUP_KEYS, dtype=object)
np.savez_compressed(os.path.join(OUT_DIR, "emb_resid.npz"), emb=resid_all, group_keys=gkeys_arr)
np.savez_compressed(os.path.join(OUT_DIR, "emb_heads.npz"), emb=heads_all, group_keys=gkeys_arr)
np.savez_compressed(os.path.join(OUT_DIR, "emb_mlp.npz"),   emb=mlp_all,   group_keys=gkeys_arr)
np.save(os.path.join(OUT_DIR, "group_real_dist.npy"), group_real_dist)
for f in ["emb_resid.npz", "emb_heads.npz", "emb_mlp.npz"]:
    sz = os.path.getsize(os.path.join(OUT_DIR, f)) / 1e6
    print(f"  {f}: {sz:.0f} MB")


## 5. Peta kesetiaan: rho per (komponen x lokasi x template x tipe)

Satu fungsi, dipakai di semua titik baca. Rho dihitung **dalam-tipe** (semua pasangan sel
di 1 tipe kombinasi) — pelajaran finding 02, pooled menyesatkan. Selain per-template, ada
varian **Tmean** (embedding dirata-ratakan antar 4 template dulu) = readout multi-cue.


In [ ]:
def within_type_rho(emb_2d, subset_idx):
    sub = emb_2d[subset_idx].astype(np.float32)
    rep = pairwise_distances(sub, metric="cosine")
    real = group_real_dist[np.ix_(subset_idx, subset_idx)]
    iu = np.triu_indices(len(subset_idx), k=1)
    a, b = real[iu], rep[iu]
    ok = ~np.isnan(a) & ~np.isnan(b)
    if ok.sum() < 5:
        return np.nan
    return float(spearmanr(a[ok], b[ok])[0])

import warnings
warnings.filterwarnings("ignore")  # spearmanr ConstantInputWarning di subset kecil

TEMPLATE_TAGS = [f"T{t}" for t in range(N_TEMPLATES)] + ["Tmean"]

def emb_at(source, t_tag, loc):
    """source: 'resid'|'mlp'|'head'; loc: layer utk resid/mlp, (layer, head) utk head."""
    if source == "resid":
        arr = resid_all[:, :, loc, :]
    elif source == "mlp":
        arr = mlp_all[:, :, loc, :]
    else:
        li, hi = loc
        arr = heads_all[:, :, li, hi, :]
    if t_tag == "Tmean":
        return arr.astype(np.float32).mean(axis=0)
    return arr[int(t_tag[1])]

rows = []

# residual: 33 lokasi
for L in tqdm(range(NUM_LAYERS + 1), desc="peta residual"):
    for t_tag in TEMPLATE_TAGS:
        e = emb_at("resid", t_tag, L)
        for ty in TYPES:
            rows.append(dict(component="resid", layer=L, head=-1, template=t_tag,
                             attr_type=ty, rho=within_type_rho(e, type_idx[ty])))

# mlp: 32 lokasi
for L in tqdm(range(NUM_LAYERS), desc="peta mlp"):
    for t_tag in TEMPLATE_TAGS:
        e = emb_at("mlp", t_tag, L)
        for ty in TYPES:
            rows.append(dict(component="mlp", layer=L, head=-1, template=t_tag,
                             attr_type=ty, rho=within_type_rho(e, type_idx[ty])))

peta_rm = pd.DataFrame(rows)
print(peta_rm.shape)


In [ ]:
# head: 32x32 = 1024 lokasi. Per-template + Tmean x 6 tipe = ~30k korelasi (CPU, beberapa menit).
rows_h = []
for L in tqdm(range(NUM_LAYERS), desc="peta head"):
    for H in range(NUM_HEADS):
        for t_tag in TEMPLATE_TAGS:
            e = emb_at("head", t_tag, (L, H))
            for ty in TYPES:
                rows_h.append(dict(component="head", layer=L, head=H, template=t_tag,
                                   attr_type=ty, rho=within_type_rho(e, type_idx[ty])))

peta_head = pd.DataFrame(rows_h)
peta = pd.concat([peta_rm, peta_head], ignore_index=True)
peta.to_csv(os.path.join(OUT_DIR, "peta_kesetiaan_full.csv"), index=False)
print("Peta lengkap:", peta.shape, "-> peta_kesetiaan_full.csv")


## 6. Baca petanya

Baseline pembanding (cara baca naif, finding 01-02): residual T0, per tipe —
AGExPOLPARTY ~0.50, EDUCATIONxINCOME ~0.49, RACExRELIG ~0.07.
Pertanyaannya: adakah lokasi yang MENGALAHKAN itu jauh — terutama buat tipe lemah?


In [ ]:
# 6a. Top-20 lokasi per tipe (pakai Tmean biar bukan artefak 1 template)
tm = peta[peta["template"] == "Tmean"].copy()
top_rows = []
print("=" * 80)
for ty in TYPES:
    sub = tm[tm["attr_type"] == ty].sort_values("rho", ascending=False)
    naive = tm[(tm["attr_type"] == ty) & (tm["component"] == "resid")]["rho"].max()
    best = sub.iloc[0]
    print(f"\n{ty}  (residual terbaik: {naive:+.3f})")
    print(sub.head(10)[["component", "layer", "head", "rho"]].to_string(index=False))
    top_rows.append(sub.head(20))
pd.concat(top_rows).to_csv(os.path.join(OUT_DIR, "peta_kesetiaan_top.csv"), index=False)


In [ ]:
# 6b. Kurva residual & mlp per layer, per tipe (Tmean)
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for comp, ax in zip(["resid", "mlp"], axes):
    sub = tm[tm["component"] == comp]
    for ty in TYPES:
        s = sub[sub["attr_type"] == ty].sort_values("layer")
        ax.plot(s["layer"], s["rho"], marker=".", label=ty)
    ax.axhline(0, color="gray", lw=0.8)
    ax.set_title(f"{comp} — rho per layer (Tmean)")
    ax.set_xlabel("layer")
axes[0].set_ylabel("Spearman rho vs jarak survei asli")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "peta_perlayer_resid_mlp.png"), dpi=150)
plt.show()


In [ ]:
# 6c. Heatmap head 32x32 per tipe (Tmean)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
hm_sub = tm[tm["component"] == "head"]
for ax, ty in zip(axes.flat, TYPES):
    grid = np.full((NUM_LAYERS, NUM_HEADS), np.nan)
    s = hm_sub[hm_sub["attr_type"] == ty]
    grid[s["layer"].values, s["head"].values] = s["rho"].values
    im = ax.imshow(grid, aspect="auto", cmap="RdBu_r", vmin=-0.6, vmax=0.6)
    ax.set_title(ty, fontsize=10)
    ax.set_xlabel("head")
    ax.set_ylabel("layer")
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6, label="rho (Tmean)")
plt.savefig(os.path.join(OUT_DIR, "peta_head_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# 6d. Sensitivitas template: di lokasi terbaik tiap tipe, seberapa goyang rho antar T0-T3?
print("Lokasi terbaik (Tmean) per tipe + rho per template — goyang = sinyal rapuh:\n")
sens_rows = []
for ty in TYPES:
    sub = tm[tm["attr_type"] == ty].sort_values("rho", ascending=False).iloc[0]
    comp, L, H = sub["component"], int(sub["layer"]), int(sub["head"])
    sel = peta[(peta["component"] == comp) & (peta["layer"] == L) &
               (peta["head"] == H) & (peta["attr_type"] == ty)]
    per_t = sel.set_index("template")["rho"]
    spread = float(per_t[[f"T{t}" for t in range(N_TEMPLATES)]].std())
    print(f"{ty:20s} {comp:5s} L{L:2d} H{H:3d} | " +
          " ".join(f"{tag}={per_t[tag]:+.2f}" for tag in TEMPLATE_TAGS) +
          f" | std antar-T: {spread:.3f}")
    sens_rows.append(dict(attr_type=ty, component=comp, layer=L, head=H,
                          std_antar_template=spread, **{k: float(v) for k, v in per_t.items()}))
pd.DataFrame(sens_rows).to_csv(os.path.join(OUT_DIR, "sensitivitas_template.csv"), index=False)


## Cara baca hasil & checklist

**Tiga pertanyaan yang dijawab peta ini:**

1. **Adakah lokasi yang jauh lebih setia dari cara baca naif?** Lihat 6a: bandingkan rho
   terbaik per tipe vs baseline residual. Kalau head/layer tertentu tembus jauh di atas
   0.50 (apalagi buat RACExRELIG yang tadinya 0.07) -> Babak 2 lanjut fokus ke situ,
   dan Babak 3 cabang positif kebuka.
2. **Sinyalnya properti model atau artefak template?** Lihat 6d: lokasi terbaik yang
   std antar-template-nya besar = rapuh, jangan dipercaya.
3. **Kalau semua lokasi tetap lemah?** Itu temuan juga (Babak 3 cabang negatif):
   identitas demografis di LLM dangkal — bisa dibaca per kelompok, tidak tertata antar
   kelompok — di SEMUA titik baca yang wajar.

**Checklist download (JANGAN dilewat):**
`peta_kesetiaan_full.csv`, `peta_kesetiaan_top.csv`, `sensitivitas_template.csv`,
kedua `.png`, `group_real_dist.npy`, dan minimal `emb_heads.npz`.

Hasil run -> paste output cell 6a & 6d ke diskusi; temuan masuk `notes/findings/05_...`.
